# 📊 S&P 500 ESG Risk Analysis

**Análisis exploratorio de riesgo ESG (Environmental, Social, Governance)** en las 503 empresas del índice S&P 500, identificando patrones sectoriales de riesgo, empresas outlier y la composición del riesgo ESG total.

**Autora:** Naiara Altuna Renedo  
**Dataset:** [S&P 500 ESG Risk Ratings (Kaggle)](https://www.kaggle.com/datasets/pritish509/s-and-p-500-esg-risk-ratings)


## ⚠️ Nota sobre la fuente y el proveedor de los datos

El dataset **no documenta explícitamente** en sus columnas qué agencia calificadora generó estos scores ESG — no existe una columna `Provider` ni metadato de atribución directa.

Por la estructura de las columnas (`Total ESG Risk score`, sub-scores de Environment/Social/Governance, `Controversy Level`, `ESG Risk Percentile`, niveles Negligible/Low/Medium/High/Severe), la metodología es **consistente con la de Sustainalytics** (subsidiaria de Morningstar), ampliamente utilizada y publicada por plataformas como Yahoo Finance para cada ticker público.

**Esta atribución es un supuesto de trabajo, no un hecho verificado contra la fuente original.** Se documenta aquí de forma explícita porque es una buena práctica de transparencia metodológica: un análisis ESG debe declarar claramente qué tan segura es la procedencia de sus datos antes de sacar conclusiones de inversión.


## 🎯 Objetivo del análisis

Como analista, simulamos el encargo de un gestor de cartera de inversión sostenible que necesita responder:

1. ¿Qué sectores del S&P 500 presentan mayor riesgo ESG en promedio?
2. ¿Existen empresas "outlier" con riesgo desproporcionado frente a su sector?
3. ¿Qué dimensión del ESG (Ambiental, Social o Gobernanza) pesa más en el riesgo total?
4. ¿Qué empresas serían candidatas a revisión prioritaria para un fondo de inversión responsable?


## 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## 2. Cargar los datos

**Si trabajas en Google Colab:** ejecuta primero la celda de carga de archivo (sube el CSV desde tu computadora). 
**Si trabajas en local/Jupyter:** comenta la celda de Colab y usa directamente `pd.read_csv()` con la ruta local.

In [ ]:
# ── Solo en Google Colab ──
# Ejecuta esta celda para subir el archivo CSV desde tu computadora.
# Si ya lo subiste antes en esta sesión, puedes saltarte este paso.

from google.colab import files
uploaded = files.upload()


In [ ]:
# Verifica el nombre exacto del archivo subido antes de leerlo
import os
print([f for f in os.listdir() if f.endswith('.csv')])


In [ ]:
# Ajusta el nombre del archivo según lo que muestre la celda anterior
df = pd.read_csv('SP 500 ESG Risk Ratings.csv')

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head()


## 3. Exploración inicial (EDA)

Antes de limpiar nada, entendemos qué tenemos: tipos de dato, valores nulos y estructura general.

In [ ]:
df.info()


In [ ]:
# Valores nulos por columna
nulos = df.isnull().sum()
nulos[nulos > 0].sort_values(ascending=False)


**Observación:** 73 de 503 empresas (≈14.5%) no tienen score ESG calculado. 
Esto es consistente con empresas que cotizan recientemente o no fueron evaluadas por el proveedor de datos en el periodo de la muestra.

## 4. Limpieza de datos

**Decisión de limpieza:** eliminamos las filas sin `Total ESG Risk score`, ya que no podemos analizar ni imputar de forma confiable un score de riesgo ESG — son evaluaciones cualitativas complejas, no un promedio estadístico simple.

In [ ]:
# Conservamos solo empresas con score ESG calculado
df_clean = df.dropna(subset=['Total ESG Risk score']).copy()

print(f"Filas originales: {len(df)}")
print(f"Filas tras limpieza: {len(df_clean)}")
print(f"Filas eliminadas: {len(df) - len(df_clean)} ({(len(df)-len(df_clean))/len(df)*100:.1f}%)")


In [ ]:
# Eliminamos la única fila sin sector (caso aislado)
df_clean = df_clean.dropna(subset=['Sector'])

# Verificación final de nulos en columnas clave
df_clean[['Total ESG Risk score', 'Environment Risk Score',
          'Social Risk Score', 'Governance Risk Score', 'Sector']].isnull().sum()


## 5. Riesgo ESG promedio por sector

¿Qué sectores son estructuralmente más riesgosos desde el punto de vista ESG?

In [ ]:
riesgo_por_sector = (
    df_clean.groupby('Sector')['Total ESG Risk score']
    .agg(['mean', 'median', 'std', 'count'])
    .sort_values('mean', ascending=False)
    .round(2)
)
riesgo_por_sector


In [ ]:
plt.figure(figsize=(10, 6))
orden_sectores = riesgo_por_sector.index

sns.boxplot(
    data=df_clean,
    x='Total ESG Risk score',
    y='Sector',
    order=orden_sectores,
    hue='Sector',
    palette='Greens_r',
    legend=False
)
plt.title('Distribución del Riesgo ESG Total por Sector — S&P 500', fontsize=13, fontweight='bold')
plt.xlabel('Total ESG Risk Score (menor = mejor)')
plt.ylabel('')
plt.tight_layout()
plt.savefig('riesgo_esg_por_sector.png', dpi=150, bbox_inches='tight')
plt.show()


**Hallazgo real:** *Energy* (32.34 promedio), *Basic Materials* (26.72) y *Utilities* (26.71) muestran el mayor riesgo ESG, consistente con la intensidad de emisiones de su actividad. *Real Estate* (13.09) y *Technology* (16.92) presentan el menor riesgo promedio del índice.

## 6. ¿Qué pesa más: Ambiental, Social o Gobernanza?

Comparamos las tres dimensiones del riesgo ESG para entender qué componente domina el score total.

In [ ]:
componentes = ['Environment Risk Score', 'Social Risk Score', 'Governance Risk Score']

medias_componentes = df_clean[componentes].mean().sort_values(ascending=False)
print(medias_componentes)

plt.figure(figsize=(7, 5))
medias_componentes.plot(kind='bar', color=['#2E7D52', '#185FA5', '#854F0B'])
plt.title('Riesgo Promedio por Componente ESG — S&P 500', fontsize=13, fontweight='bold')
plt.ylabel('Score promedio de riesgo')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('componentes_esg_promedio.png', dpi=150, bbox_inches='tight')
plt.show()


**Hallazgo clave:** el riesgo **Social** (9.07 promedio) supera al de **Gobernanza** (6.73) y **Ambiental** (5.74). La narrativa pública sobre ESG suele centrarse en lo ambiental, pero en este índice el componente social tiene mayor peso relativo en el riesgo total.

In [ ]:
# Correlación entre los tres componentes
correlacion = df_clean[componentes + ['Total ESG Risk score']].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(correlacion, annot=True, cmap='Greens', fmt='.2f', square=True)
plt.title('Correlación entre componentes del riesgo ESG', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('correlacion_componentes_esg.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Detección de empresas outlier

Identificamos empresas cuyo riesgo ESG se desvía significativamente del promedio de su propio sector — son las que un analista de inversión sostenible revisaría primero.

In [ ]:
# Calculamos el z-score del riesgo ESG dentro de cada sector
df_clean['z_score_sector'] = df_clean.groupby('Sector')['Total ESG Risk score'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# Outliers: más de 1.5 desviaciones estándar por encima de su sector
outliers = df_clean[df_clean['z_score_sector'] > 1.5].sort_values('z_score_sector', ascending=False)

print(f"Número de empresas outlier: {len(outliers)} de {len(df_clean)}")
outliers[['Name', 'Sector', 'Total ESG Risk score', 'ESG Risk Level', 'z_score_sector']].round(2)


**Interpretación:** estas empresas presentan un riesgo ESG notablemente superior al promedio de su propio sector — no son simplemente las peores en términos absolutos, sino las que más se desvían de lo esperado para su industria. Esa distinción es clave: un banco con riesgo 25 puede ser un outlier grave, mientras que una petrolera con riesgo 25 puede estar dentro de lo normal para su sector.

**Resultado real:** 30 empresas (de 430 con score válido) califican como outlier. Destacan **Wells Fargo & Co.** y **Fortive Corporation**, con desviaciones superiores a 3 desviaciones estándar frente a su sector.

## 8. Ranking: mejores y peores empresas en gestión ESG

In [ ]:
top_10_mejores = df_clean.nsmallest(10, 'Total ESG Risk score')[
    ['Name', 'Sector', 'Total ESG Risk score', 'ESG Risk Level']
]
print("🟢 TOP 10 — Menor riesgo ESG (mejor gestión)")
top_10_mejores


In [ ]:
top_10_peores = df_clean.nlargest(10, 'Total ESG Risk score')[
    ['Name', 'Sector', 'Total ESG Risk score', 'ESG Risk Level']
]
print("🔴 TOP 10 — Mayor riesgo ESG (requieren atención)")
top_10_peores


## 9. Conclusiones — Informe ejecutivo

**Para:** Equipo de inversión sostenible  
**De:** Análisis de Riesgo ESG, S&P 500

1. **Riesgo por sector:** los sectores de energía y materiales básicos (*Energy*, *Basic Materials*, *Utilities*) muestran consistentemente mayor riesgo ESG promedio, reflejando la intensidad de emisiones y regulación ambiental de su actividad. *Real Estate* y *Technology* presentan el menor riesgo.

2. **Componente dominante:** el riesgo **Social** (9.07) supera tanto al de Gobernanza (6.73) como al Ambiental (5.74) — un hallazgo relevante porque la narrativa pública suele centrarse solo en lo ambiental.

3. **Empresas outlier:** se identificaron 30 empresas con riesgo ESG significativamente superior al promedio de su propio sector — candidatas prioritarias para revisión de un comité de inversión responsable.

4. **Recomendación:** cualquier estrategia de screening ESG debería normalizar por sector antes de comparar empresas, ya que el riesgo "bueno" o "malo" depende fuertemente de la industria de referencia.

5. **Limitación metodológica:** el proveedor exacto de estos scores no está confirmado en el dataset original (ver nota al inicio del notebook). Cualquier uso de estos datos para decisiones reales de inversión debería validar la fuente directamente con Sustainalytics o el proveedor correspondiente.

---
*Próximos pasos: cruzar este dataset con datos financieros (rentabilidad, volatilidad) para evaluar si existe relación entre riesgo ESG y desempeño bursátil.*
